# Baseline — Benign-Only Autoencoder

Trains the autoencoder on benign-only rows from `data/processed/train.csv`, picks a detection threshold from benign `val` scores at a target false-positive rate, and reports metrics on `val` as a pilot checkpoint.

This notebook assumes `data/processed/{train,val,test}.csv` already exist — run `Milestone1_Draft.ipynb` first if they don't, or if you've changed the data split.

In [12]:
import sys
from pathlib import Path

# Make the project root importable regardless of PyCharm's working directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project


## Train, threshold, and evaluate on val

`run_pilot()` does all of it: trains on train-benign, picks the threshold on val-benign at a 1% target FPR, evaluates on the full val split, and saves the model/scaler/threshold to `models/` for reuse.

In [13]:
from src.autoencoder import run_pilot

model, scaler, threshold, val_metrics = run_pilot(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    target_fpr=0.01,
    models_dir=PROJECT_ROOT / "models",
)

  dropped 465 rows with inf/NaN feature values
  dropped 85 rows with inf/NaN feature values
Training on 643549 benign rows (train split has 779176 rows total, 135627 attack)
Training autoencoder...
  epoch   1/30  train MSE: 0.270619
  epoch   5/30  train MSE: 0.064848
  epoch  10/30  train MSE: 0.037865
  epoch  15/30  train MSE: 0.029044
  epoch  20/30  train MSE: 0.030895
  epoch  25/30  train MSE: 0.028539
  epoch  30/30  train MSE: 0.022117

Threshold set at 1.0% target FPR on benign val scores: 0.079475

=== Baseline (autoencoder) — val split (pilot checkpoint) ===
  threshold : 0.0795
  accuracy  : 0.9663
  precision : 0.9473
  recall    : 0.8541
  f1        : 0.8983
  fpr       : 0.0100
  fnr       : 0.1459
  n         : 166978
  n_attack  : 29061
  n_benign  : 137917
Saved model, scaler, threshold to /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/models/


## Sanity check

val_metrics is already printed above by run_pilot(), but here it is as a dict for quick reference or to compare later against RF-IDS.

In [14]:
val_metrics

{'threshold': 0.07947487384080887,
 'accuracy': 0.9663368827031106,
 'precision': 0.9473282442748091,
 'recall': 0.8540655861807921,
 'f1': 0.8982826948480845,
 'fpr': np.float64(0.01000601811234293),
 'fnr': np.float64(0.14593441381920788),
 'n': 166978,
 'n_attack': 29061,
 'n_benign': 137917}

## Final test-set score (run ONCE, at the end)

Don't run this cell while you're still iterating on the model above — it's meant to be called exactly once, after the model and threshold are frozen, for the real milestone comparison. Running it repeatedly turns the test set into development data, which is exactly the leakage problem the instructor's feedback flagged.

In [15]:
# from src.autoencoder import score_test
# test_metrics = score_test(
#     processed_dir=PROJECT_ROOT / "data" / "processed",
#     models_dir=PROJECT_ROOT / "models",
# )